# 03 - Backtest (Multi-Asset Use Case)

This notebook demonstrates a practical portfolio workflow using **NVDA, AMZN, AAPL**:

1. Load bars for each symbol with `yfinance`.
2. Merge into timestamp baskets compatible with the multi-asset engine.
3. Run a basket strategy that emits per-symbol target weights.
4. Evaluate equity curve, fills, and per-symbol exposure.


In [ ]:
from collections import defaultdict, deque

import matplotlib.pyplot as plt
import pandas as pd

from analytics.tearsheet import make_tearsheet_with_details
from data.loaders.yfinance_loader import load_yfinance
from engine.core.backtest_engine import BacktestEngine, EngineConfig
from engine.core.event import MarketEvent


## 1) Configure universe and engine

`max_abs_weight` limits per-symbol exposure, while fees/slippage/spread model execution realism.


In [ ]:
SYMBOLS = ["NVDA", "AMZN", "AAPL"]
PERIOD = "2y"
INTERVAL = "1d"

engine_cfg = EngineConfig(
    initial_cash=100_000,
    fee_bps=1.0,
    slippage_bps=2.0,
    spread_bps=1.0,
    latency_bars=0,
    max_abs_weight=0.6,
)


## 2) Load and align bars

The engine expects a flat `list[MarketEvent]` sorted by timestamp with symbol-tagged bars.


In [ ]:
def load_universe(symbols: list[str], period: str, interval: str) -> list[MarketEvent]:
    bars = []
    for symbol in symbols:
        bars.extend(load_yfinance(symbol=symbol, period=period, interval=interval, auto_adjust=True))

    bars.sort(key=lambda b: (b.timestamp, b.symbol))
    return bars


bars = load_universe(SYMBOLS, PERIOD, INTERVAL)
print(f"Loaded {len(bars)} bars across {len(SYMBOLS)} symbols")
print("Date range:", bars[0].timestamp, "->", bars[-1].timestamp)


## 3) Define a basket strategy (ranked momentum, long-only)

- Compute simple lookback return per symbol.
- Allocate only to the top-ranked symbol (long-only).
- Keep other symbols at 0 target weight.


In [ ]:
class RankedMomentumBasket:
    def __init__(self, lookback: int = 20, gross_target: float = 1.0):
        self.lookback = lookback
        self.gross_target = gross_target
        self.history: dict[str, deque[float]] = defaultdict(lambda: deque(maxlen=lookback + 1))

    def on_bars(self, bars_by_symbol: dict[str, MarketEvent]) -> dict[str, float]:
        rets = {}
        for symbol, bar in bars_by_symbol.items():
            h = self.history[symbol]
            h.append(bar.close)
            if len(h) < self.lookback + 1:
                continue
            rets[symbol] = (h[-1] / h[0]) - 1.0

        # warmup: stay flat
        if len(rets) < 2:
            return {symbol: 0.0 for symbol in bars_by_symbol}

        ranked = sorted(rets.items(), key=lambda kv: kv[1])
        winner = ranked[-1][0]

        targets = {symbol: 0.0 for symbol in bars_by_symbol}
        targets[winner] = min(1.0, self.gross_target)
        return targets


## 4) Run backtest


In [ ]:
engine = BacktestEngine(engine_cfg)
strategy = RankedMomentumBasket(lookback=20, gross_target=1.0)
result = engine.run_detailed(bars, strategy)

print("Equity points:", len(result.equity_curve))
print("Fills:", len(result.fills))
print("Final cash:", round(engine.portfolio.state.cash, 2))
print("Final positions:", {k: round(v, 4) for k, v in engine.portfolio.state.positions.items()})


## 5) Inspect performance summary


In [ ]:
# one representative close per timestamp for analytics alignment
close_by_ts = {}
for bar in bars:
    close_by_ts.setdefault(bar.timestamp, bar.close)
prices = list(close_by_ts.values())[: len(result.positions)]

metrics = make_tearsheet_with_details(
    result.equity_curve,
    annualization=252,
    fills=result.fills,
    positions=result.positions,
    prices=prices,
    benchmark_equity=None,
    rolling_window=20,
)

pd.Series(metrics).sort_index()


## 6) Visual diagnostics


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)

pd.Series(result.equity_curve).plot(ax=axes[0], title="Portfolio equity curve")
axes[0].set_ylabel("Equity")

pd.Series(result.cash_series).plot(ax=axes[1], title="Cash balance")
axes[1].set_ylabel("Cash")

plt.tight_layout()


## 7) Fills breakdown by symbol


In [ ]:
fills_df = pd.DataFrame([
    {
        "timestamp": f.timestamp,
        "symbol": f.symbol,
        "side": f.side,
        "qty": f.quantity,
        "price": f.fill_price,
        "fee": f.fee,
    }
    for f in result.fills
])

if fills_df.empty:
    print("No fills yet (possibly still in warmup).")
else:
    display(fills_df.head())
    display(fills_df.groupby(["symbol", "side"]).size().rename("count"))
